# 基于 TPipe/TQue 框架的 SIMD 编程

前面我们已经了解了 Ascend C SIMD 编程的基础概念。本节围绕 `TPipe` 和 `TQue` 展开，学习如何使用TPipe/TQue 框架完成算子编程。本文以最基础的矢量算子编程范式为例，展示在 AI Core 上的数据搬入、矢量计算、结果搬出，以及如何使用TPipe/TQue框架处理流水任务之间的通信与同步。

### 前置要求

学习本节前，建议你已经了解以下内容：

- 掌握 C/C++ 函数声明、指针参数、引用参数和模板函数的基本语法；
- 了解 NPU的基础硬件架构知识；
- 具备入门级的AscendC算子编写能力。

### 本节学习目标如下：

- 了解 AI Core 流水线并行的基本思想；
- 了解什么是流水同步；
- 了解 `TPipe` 和 `TQue` 在 AscendC编程范式中的作用；
- 掌握 `InitBuffer`、`AllocTensor`、`EnQue`、`DeQue`、`FreeTensor` 等核心接口的使用方法；
- 能够基于 Add 样例看懂并编写一个完整的 `TPipe/TQue` 矢量算子程序。

### 本节学习大纲如下：

- 环境准备；
- 矢量算子编程范式；
- 基于 `TPipe/TQue` 框架的矢量算子编程；
- `TPipe/TQue` 框架编码使用的接口介绍；
- 矢量算子编程实践；
- Q&A；
- 本章总结；
- 课后习题。


---

## 1. 环境准备

正式开始学习之前，先对 Jupyter 环境进行初始化。以下代码会创建本节使用的 `TPipeSources` 目录，并将 CANN 环境变量导入当前 Notebook 进程，便于后续编译或查看样例代码。


In [ ]:
import os
import subprocess
from pathlib import Path

Path("TPipeSources").mkdir(exist_ok=True)

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")


---

## 2. 矢量算子编程范式

### 2.1 Ascend C 编程范式

为了充分发挥硬件的极致算力，AI Core 内部采用了流水线并行执行架构。多个执行单元可以同时工作，并以流水作业的方式完成完整的算子执行过程。

如下流水线并行示意图展示了这一概念。一个数据分片从输入到输出需要经过多个阶段任务（T1、T2、T3），多个执行单元可以并行处理不同阶段。每个执行单元专注于一类任务，并处理所有数据分片；当某个执行单元完成当前分片后，会将结果交给下一个阶段继续处理。这个过程可以类比为生产流水线：每个工人只负责固定工序，产品在工序之间流转，从而提升整体吞吐。

<img src="./images/03.03.06_pipeline_parallelism_schematic.png" alt="pipeline_parallelism_schematic" width="700px">

如何清晰地定义各工序？如何安全、高效地在工序间传递半成品数据？如何管理整个流水线共享的资源？

Ascend C 编程范式正是为解决这些问题而设计的流水线式编程范式。它将一个 Ascend C 算子核函数内的实现逻辑划分为多个流水任务，对应流水线上的不同工序，再定义连接这些任务的通信队列，最后实现每个任务的具体逻辑。这套范式屏蔽了大量底层硬件并行与同步细节，使开发者能够快速搭建算子实现框架，并发挥 AI Core 的流水线并行能力。

### 2.2 基本的矢量算子编程范式

在 Ascend C 的编程实践中，根据算子计算逻辑与数据流复杂度的不同，主要有基本矢量编程范式、基本矩阵编程范式和复杂矢量/矩阵编程范式三种典型计算范式，它们构成了构建算子的核心模板。

本节从最基础、最常用的基本矢量编程范式开始介绍。该范式将计算流程清晰地划分为三个基本流水任务：CopyIn（数据搬入）、Compute（矢量计算）、CopyOut（结果搬出）。它结构简单，是理解流水线编程思想的起点。

<img src="./images/03.03.06_vector_programming_paradigm.png" alt="vector_programming_paradigm"  width="700px" >

各流水任务的具体职责如下：

- **CopyIn** 负责搬入操作：

    将输入数据从 Global Memory 搬运到 Local Memory（Unified Buffer）。`VECIN` 用于表达矢量计算搬入数据的存放位置。搬运完成后，需要通知下游数据已经准备好，将数据递交给下游任务。

- **Compute** 负责矢量计算：

    从输入队列中取得上游已经准备好的数据，在 Local Memory 中调用矢量计算接口完成核心计算。计算完成后，需要把结果放入输出队列，通知下游任务可以继续处理。

- **CopyOut** 负责搬出操作：

    从输出队列中取得计算结果，将结果从 Local Memory 搬运回 Global Memory，`VECOUT`用于表达矢量计算搬出数据的存放位置，完成算子的最终输出。

### 2.3 流水同步问题

以 Add 矢量算子编程范式为例，单个数据分片需要完成如下流水任务：

```cpp
// 1. CopyIn: 把 x、y 从 GM 搬到 UB
AscendC::DataCopy(xLocal, xGm, blockLength);
AscendC::DataCopy(yLocal, yGm, blockLength);

// 2. Compute: 在 UB 中执行 z = x + y
AscendC::Add(zLocal, xLocal, yLocal, blockLength);

// 3. CopyOut: 把 z 从 UB 搬回 GM
AscendC::DataCopy(zGm, zLocal, blockLength);
```

需要注意的是，在 AI Core 上，CopyIn、Compute、CopyOut 通常对应不同的硬件执行单元。代码虽然按顺序书写，但这些接口更接近于向不同硬件执行单元发射任务，而不是普通 CPU 程序中“上一行完全执行完，下一行才开始”的简单同步语义。

这就引出了流水编程的关键问题：

第一类是**上游写完之前，下游不能读**。

例如 `DataCopy(xLocal, xGm, len)` 只是向搬运单元发出了搬入任务，后面的 `Add(zLocal, xLocal, yLocal, len)` 不能在 `xLocal/yLocal` 还没有真正搬完时就开始读。否则 Vector 单元可能读到未就绪的数据。

第二类是**下游读完之前，上游不能复用同一块 buffer 再写**。

由于UB 空间有限，下一轮的数据分片通常会复用上一轮的 `xLocal` 或 `zLocal` buffer。如果上一轮 `Add` 还在读 `xLocal`，下一轮搬入的`DataCopy(xLocal, xGm, len)` 就提前覆盖 `xLocal`，计算就会出错；如果上一轮搬出的`DataCopy(zGm, zLocal, len)` 还在读 `zLocal`，下一轮 `Add` 就提前写 `zLocal`，搬出的结果也会被破坏。

所以，在 AI Core 异步流水里，开发者必须同时处理两件事：

- **阶段间时序同步**：保证上游写完，下游再读。
- **内存复用同步**：保证旧消费者读完，新生产者再覆盖。


---

## 3. 基于 `TPipe` / `TQue` 框架的矢量算子编程

### 3.1 手动实现流水同步和内存控制的痛点

为了处理流水时序同步和内存复用，开发者可以通过插入 `SetFlag/WaitFlag`手动安排底层同步事件：

- SetFlag：当源流水的前序指令的所有读写操作都完成之后，当前指令开始执行，并将硬件中的对应标志位设置为1。SetFlag只是设置硬件中的对应标志位，并不会阻塞源流水中的下一个指令；
- WaitFlag：当目的流水执行到该指令时，如果发现硬件中对应标志位为0，目的流水的后续指令将一直被阻塞；如果发现硬件中对应标志位为1，则将硬件中对应标志位设置为0，同时目的流水的后续指令开始执行。

但是如果让开发者直接通过 `SetFlag/WaitFlag`手动安排底层同步事件，保证阶段间流水同步并维护每块 buffer 当前到底被哪个执行单元读写，代码会变得复杂且容易出错：

- 同步点少了，可能读到尚未搬入完成的数据，或者提前覆盖仍在被读取的 buffer；
- 同步点多了，本来可以并行的 CopyIn、Compute、CopyOut 会被强行串行化，导致性能下降；
- 代码会被大量底层同步细节淹没，真正的算子计算逻辑反而不清晰。

### 3.2 什么是 `TPipe/TQue`

为降低上述同步和内存管理成本，Ascend C 引入了 `TPipe/TQue` 框架：

- `TPipe` 是资源管理器，用于统一管理队列 buffer、临时内存以及同步事件等 Device 端资源；
- `TQue` 是队列，用于在 CopyIn、Compute、CopyOut 等任务阶段之间传递 `LocalTensor`，并在底层配合同步事件保证阶段间时序正确。

具体来说，`TPipe/TQue` 把两类底层问题封装成更接近队列操作的接口：

- **使用`EnQue`和`DeQue`解决阶段间时序同步问题**，保证上游数据写完之后，下游才读取。具体而言：上游将EnQue作为生产者信号，调用EnQue后底层自动发射SetFlag硬件同步指令，该流水前序指令执行完成后，SetFlag指令会标记当前阶段写入已完成，并主动唤醒下游阻塞任务。下游调用DeQue后底层发射WaitFlag同步指令，WaitFlag指令会让下游任务进入阻塞，直至收到上游SetFlag唤醒信号，确保所需数据完全就绪后下游才执行读取。

<img src="./images/03.03.06_Iterate_flase_2_enque.png" alt="任务时序示意图" width="820px">

- **使用`AllocTensor`和`FreeTensor`解决内存复用问题**，保证内存空间上的旧数据不会再被读写之后，再覆盖旧数据。具体而言：当任务调用`AllocTensor`申请某块内存时，底层发射WaitFlag同步指令，持续等待该内存区域的所有读取操作彻底完成、进入可覆写状态，随后才分配内存并解除任务阻塞。当任务使用完毕并调用`FreeTensor`时，底层发射SetFlag同步指令，通知硬件该内存已无读取依赖，可以进入可覆写状态，供下一轮迭代任务复用，所以每轮迭代必须`FreeTensor`，保证下一轮迭代`AllocTensor`的时候可以申请到内存，否则下一轮`AllocTensor`会一直等待。

<img src="./images/03.03.06_Iterate_flase_alloc.png" alt="任务时序示意图" width="820px">

`TPipe/TQue` 把基础流水中的资源管理、队列通信和同步关系封装起来，就可以避免让用户在基础流水中直接编排大量底层同步事件。算子的不同任务阶段之间通过专门的队列（TQue） 进行通信和同步，确保数据像在传送带上一样，从一个任务平稳地交付到下一个任务。同时，通过统一的资源管理模块（TPipe），对任务间共享的内存、事件等资源进行生命周期管理和调度，保障整条流水线顺畅、无冲突地运转。

### 3.3 基于 `TPipe` / `TQue` 框架的矢量算子编程

接下来介绍如何按照固定的矢量算子编程范式组织 CopyIn、Compute、CopyOut 三个流水阶段，利用 `TPipe/TQue` 的资源管理和同步功能写好算子。具体而言，基于 `TPipe/TQue` 框架的矢量编程如下图所示：

<img src="./images/03.03.06_queue.png" alt="queue" width="700px">

- **CopyIn**：申请输入队列中的 `LocalTensor`，使用 `DataCopy` 将输入数据从 Global Memory 搬运到 Local Memory；搬运任务发射后，通过 `EnQue` 将该 tensor 递交给下游 Compute 阶段。

- **Compute**：通过 `DeQue` 从输入队列取得上游已经准备好的 `LocalTensor`，在 Local Memory 中调用 `Add` 等矢量计算接口完成核心计算；计算结果写入输出队列申请到的 `LocalTensor`，再通过 `EnQue` 递交给下游 CopyOut 阶段。输入 tensor 使用完毕后，通过 `FreeTensor` 归还给输入队列。

- **CopyOut**：通过 `DeQue` 从输出队列取得计算结果，使用 `DataCopy` 将结果从 Local Memory 搬运回 Global Memory；搬出任务发射后，通过 `FreeTensor` 归还输出 tensor 对应的队列 buffer。

在这套流程中，`TQue` 的 `EnQue/DeQue` 主要用于表达流水阶段之间的交接关系，并在底层配合同步事件保证读写顺序；`TPipe` 则统一管理队列 buffer、事件等 Device 端资源。底层的同步关系由 `TPipe/TQue` 管理，用户只需要按照基于 `TPipe/TQue` 的矢量编程范式固定流程编写代码，就可以实现一个流水同步正确的基础矢量算子。

### 3.4 `TPipe/TQue` 框架的局限性

`TPipe/TQue` 的价值在于降低基础流水编程中的同步和内存管理成本，但它并不等价于“自动得到最优性能”。框架本身也有一些性能局限性，实际开发中需要注意复杂高性能场景可能需要更精细的手动设计：

当算子对极致吞吐、UB 占用、bank 冲突、GM 对齐、搬运粒度等非常敏感时，仅依赖默认 `TPipe/TQue` 写法可能无法达到最优性能，需要结合底层硬件特点做专门优化。

因此，可以把 `TPipe/TQue` 理解为基础 SIMD 流水编程的通用安全框架：它优先解决同步正确性、资源生命周期和代码结构问题；当目标是极致性能时，还需要在这个基础上继续做流水编排、内存复用和数据搬运优化。

---

## 4. `TPipe/TQue` 框架编码使用的接口介绍

理解基础矢量编程范式之后，需要从编码角度介绍编程范式里使用接口的基本用法，每个阶段用到的接口如下：

| 阶段 | 调用的接口 |
| --- | --- |
| Init：内存初始化、创建队列资源 | `TPipe`、`TQue`、`InitBuffer` |
| CopyIn：数据搬入 | `AllocTensor`、`EnQue` |
| Compute：核心计算 | `DeQue`、Ascend C 编程类库 API、`EnQue`  |
| CopyOut：结果搬出 | `DeQue` 、`FreeTensor`|

### 4.1 `TPipe`

`TPipe` 用于统一管理 Device 端内存等资源。**一个 Kernel 函数必须且只能初始化一个 `TPipe` 对象**。常见做法是在 Kernel 入口创建 `TPipe` 对象，使其生命周期覆盖整个 Kernel 函数执行过程：

```cpp
__global__ __vector__ void add_custom(...)
{
    AscendC::TPipe pipe;
    ...
}
```

### 4.2 `TQue`

`TQue` 的职责是：流水任务之间通过队列完成通信和同步。`TQue` 的模板形式如下：

```cpp
template <TPosition pos, int32_t depth, auto mask = 0>
class TQue { ... };
```

其中：

- `pos` 指队列位置，表示队列服务于哪类流水阶段或哪类存储位置，例如 `VECIN` 表示矢量计算输入队列，`VECOUT` 表示矢量计算输出队列；
- `depth` 指队列深度， 队列可连续入队/出队的深度，也就是连续 `EnQue` 中间没有 `DeQue` 时需要容纳的次数

### 4.3 `InitBuffer`

`InitBuffer` 用于为队列申请片上 buffer，常用原型如下：

```cpp
template <class T>
__aicore__ inline bool InitBuffer(T& que, uint8_t num, uint32_t len)
```

`InitBuffer` 是 `TPipe` 的接口，用于为 `TQue` 等队列对象分配片上 buffer。

| 参数 | 含义 |
|---|---|
| `T` | 队列对象类型，支持 `TQue`、`TQueBind`、`TSCM` 类型，常见为 `TQue` |
| `que` | 需要分配内存的队列对象 |
| `num` | 分配的 buffer 块数|
| `len` | 每块 buffer 的大小，单位是字节；如果不满足 32 字节对齐，接口内部会向上补齐，实际开发时仍建议主动按数据块和硬件搬运粒度设计对齐，避免非对齐处理带来的额外成本。 |

### 4.4 `AllocTensor`

`AllocTensor` 是 `TQue` 的接口，用于从队列已经初始化好的 buffer 中申请一个 `LocalTensor`。为了避免一块片上内存仍有读者访问时就被新数据覆盖，将导致未读完数据丢失、计算出错，当任务使用`AllocTensor`申请某块内存时，底层发射WaitFlag同步指令，持续等待该内存区域的所有读取操作彻底完成、进入可覆写状态，随后才分配内存并解除任务阻塞。常用原型如下：

```cpp
template <typename T> __aicore__ LocalTensor<T> AllocTensor();
```

示例：

```cpp
AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
```

如果队列中暂时没有空闲 buffer，`AllocTensor` 会在底层发射 Wait 同步指令，等待对应 buffer 重新可用。

### 4.5 `FreeTensor`

`FreeTensor` 释放队列中的指定 tensor及其占用的内存。当指定 tensor使用完毕，后续不会再读取该tensor，调用`FreeTensor`，底层发射SetFlag同步指令，通知硬件该内存已无读取依赖，可以安全释放或覆写，供后续任务复用。`FreeTensor`函数原型如下：

```cpp
template <typename T> __aicore__ void FreeTensor(const LocalTensor<T>& tensor);
```

示例：

```cpp
inQueueX.FreeTensor(xLocal);
```

`FreeTensor` 不是简单地在 CPU 语义下立刻释放数据，而是将 buffer 归还给队列，允许后续任务通过 `AllocTensor` 复用这块 buffer。

### 4.6 `EnQue`

`EnQue` 可以理解为生产者信号，作为上游任务的收尾操作，调用后把 tensor 放入队列，底层自动发射SetFlag硬件同步指令，标记上游任务已经完成该 tensor 对应的生产阶段，并主动唤醒下游阻塞任务。常用原型如下：

```cpp
template <typename T> __aicore__ void EnQue(const LocalTensor<T>& tensor);
```

示例：

```cpp
inQueueX.EnQue(xLocal);
```

在底层，`EnQue` 会配合 Set 同步事件通知下游任务：该xLocal tensor 已经可以被消费。

### 4.7 `DeQue`

`DeQue` 用于从队列中取出一个已经就绪的 `LocalTensor`。`DeQue`可以理解为消费者等待，作为下游任务的起始操作，在tensor `EnQue`操作之后，调用`DeQue`底层发射WaitFlag同步指令，当前任务进入阻塞，直至收到上游SetFlag信号，确保所需数据完全就绪后才执行读取队列里的 tensor。常用原型如下：

```cpp
template <typename T> __aicore__ LocalTensor<T> DeQue();
```

示例：

```cpp
AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
```

如果队列为空，`DeQue` 会等待上游任务执行 `EnQue`。在 tensor 执行 `EnQue` 操作之后，调用 `DeQue` 会在底层配合 WaitFlag 同步事件，确保下游任务读取该 tensor 前，上游写入或搬运已经完成。

### 4.8 Ascend C 编程类库 API

Ascend C 编程类库 API是Ascend C提供的一组类库API，提供GlobalTensor和LocalTensor等基础数据结构、数学库、Matmul、Softmax等API，支持开发者高效实现算子开发与性能优化。

`Compute`阶段时，可以调用Ascend C提供类库API完成特定的算子功能，提升开发效率。

---

## 5. 矢量算子编程实践

本小节以 Add 算子为例，展示一个基于 `TPipe/TQue` 的矢量算子编程实践样例。

### 5.1 算子分析

在编写矢量算子程序前，首先需要明确算子的计算逻辑、数据需求以及所需调用的 Ascend C 编程接口。下面以 Add 算子为例，展示这一分析过程。

1. 明确算子的数学表达式及计算逻辑。

    Add 算子的数学表达式为：

    $\vec{z} = \vec{x} + \vec{y}$

    计算逻辑是输入数据需要经历“搬运、计算、搬运”三个阶段：

    - 将输入数据从 Global Memory 搬运至 Local Memory；
    - 在 Local Memory 中使用 Ascend C 计算接口执行矢量加法；
    - 将计算结果从 Local Memory 搬运回 Global Memory。

2. 明确输入和输出。

    - 输入数量：2 个，分别为 `x` 和 `y`；
    - 输出数量：1 个，为 `z`；
    - 数据类型：输入与输出均为 `float` 类型；
    - 数据形状（Shape）：输入 Shape 为 `(8, 2048)`，输出 Shape 与输入相同。

3. 确定核函数名称和参数。

    根据输入输出分析，确定核函数名称和参数，并定义核函数原型：

    - 核函数名称：`add_custom`；
    - 参数列表：
      - `x`、`y`：输入数据在 Global Memory 上的起始地址；
      - `z`：输出数据在 Global Memory 上的起始地址。

基于以上算子分析，再结合矢量编程范式，可以把 Add 算子的实现拆分为以下三个流水任务：

<img src="./images/03.03.06_vector_programming_paradigm_and_api.png" alt="vector_programming_paradigm_and_api"  width="700px">

- Stage 1：**CopyIn**
    1. 使用`AllocTensor`给输入数据分配空间
    2. 使用 `DataCopy` 接口将输入 `GlobalTensor` 拷贝到 `LocalTensor`；
    3. 使用 `EnQue` 将该 `LocalTensor` 放入 `VECIN` 位置的队列中，通知 Compute 任务数据就绪。

- Stage 2：**Compute**

    1. 使用 `DeQue` 从 `VECIN` 队列中取出 `LocalTensor`；
    2. 使用 `Add` 接口完成矢量加法；
    3. 使用 `FreeTensor` 释放输入 `LocalTensor` 对应的 buffer。
    4. 使用 `EnQue` 将计算结果放入 `VECOUT` 位置的队列中，通知 CopyOut 任务结果就绪；

- Stage 3：**CopyOut**

    1. 使用 `DeQue` 从 `VECOUT` 队列中取出结果 `LocalTensor`；
    2. 使用 `DataCopy` 接口将计算结果从 `LocalTensor` 拷贝回输出 `GlobalTensor`；
    3. 使用 `FreeTensor` 释放输出 `LocalTensor` 对应的 buffer。

### 5.2 init阶段

介绍完Add算子的详细矢量编程流程之后，接下来开始编写Add矢量算子程序。

首先，写入编写核函数和main函数需要的头文件：


In [ ]:
%%writefile TPipeSources/add_tpipe_tque.asc

#include <algorithm>
#include <cstdint>
#include <iostream>
#include <iterator>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"

写入核函数声明：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

__global__ __vector__ void add_custom(__gm__ uint8_t* x, __gm__ uint8_t* y, __gm__ uint8_t* z,
    uint32_t totalLength)
{

接着在核函数入口声明一个 `TPipe` 对象用于管理资源，并声明三个 `TQue` 对象用于流水任务之间的通信与同步：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueueZ;

然后准备描述输入、输出数据的 `GlobalTensor` 对象，并根据当前 block 计算本核函数实例需要处理的数据范围：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;

    uint32_t blockLength = totalLength / AscendC::GetBlockNum();
    uint32_t offset = blockLength * AscendC::GetBlockIdx();
    xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
    yGm.SetGlobalBuffer((__gm__ float*)y + offset, blockLength);
    zGm.SetGlobalBuffer((__gm__ float*)z + offset, blockLength);

通过以上两步，我们已经准备好输入数据和用于管理资源的 `pipe` 对象。接下来使用 `InitBuffer` 申请队列 buffer，为 `inQueueX`、`inQueueY`、`outQueueZ` 各分配 1 块大小为 `blockLength * sizeof(float)` 字节的 UB buffer：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    pipe.InitBuffer(inQueueX, 1, blockLength * sizeof(float));
    pipe.InitBuffer(inQueueY, 1, blockLength * sizeof(float));
    pipe.InitBuffer(outQueueZ, 1, blockLength * sizeof(float));



### 5.3 CopyIn阶段


接下来使用 `AllocTensor` 从输入队列中申请 `LocalTensor`。`xLocal` 和 `yLocal` 分别绑定 `inQueueX`、`inQueueY` 中的一块 UB buffer，用于接收从 Global Memory 搬入的数据：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();

随后将 GM 中的数据搬运到刚申请的 UB buffer 中：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    AscendC::DataCopy(xLocal, xGm, blockLength);
    AscendC::DataCopy(yLocal, yGm, blockLength);

数据搬入任务发射后，需要使用 `EnQue` 将 `xLocal` 和 `yLocal` 放入输入队列。它表示 CopyIn 阶段已经把这两个 tensor 作为产物递交给下游 Compute 阶段：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);



### 5.4 Compute阶段

CopyIn 阶段把数据搬到 UB 之后，Compute 阶段需要使用 `DeQue` 从输入队列中取出 `xLocal` 和 `yLocal`：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    xLocal = inQueueX.DeQue<float>();
    yLocal = inQueueY.DeQue<float>();

获得输入数据之后，Compute阶段可以调用AscendC类库API完成特定功能的计算，根据对数据操作方法的不同，AscendC类库API分为以下两类：
- **连续计算API**：支持Tensor前n个数据计算。针对源操作数的连续n个数据进行计算并连续写入目的操作数，解决一维tensor的连续计算问题。
- **高维切分API**：支持Repeat和Stride。功能灵活的计算API，提供与BuilitIn API完全对等编程能力，充分发挥硬件优势，支持对每个操作数的DataBlock Stride，Repeat Stride，Mask等参数的操作。

本节 Add 样例使用连续计算AscendC::Add完成逐元素相加：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    AscendC::Add(zLocal, xLocal, yLocal, blockLength);

将Add指令发射到计算单元之后，使用`FreeTensor` 释放`xLocal` 和 `yLocal`占用的队列 buffer，`FreeTensor`底层的SetFlag会等到计算单元计算完成之后再将`xLocal` 和 `yLocal`对应的 buffer 归还给队列：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);

将Add指令发射到计算单元之后，也需要将结果 tensor 放入输出队列，EnQue会等到计算单元计算完成之后通知 CopyOut 阶段：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    outQueueZ.EnQue<float>(zLocal);



### 5.5 CopyOut阶段

随后从输出队列 `outQueueZ` 中取出计算结果，使用 `DataCopy` 将 `zLocal` 中的数据搬回 GM：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm, zLocal, blockLength);

存放输出结果的 tensor 使用完毕后，使用 `FreeTensor` 释放该 tensor 对应的队列 buffer：

In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

    outQueueZ.FreeTensor(zLocal);
}

通过以上实现，我们完成了基于 `TPipe/TQue` 的 Add 核函数代码。这个过程体现了两点核心思想：

1. 三级流水线：CopyIn、Compute、CopyOut 任务通过队列协同工作；
2. 统一资源管理：`TPipe` 负责队列 buffer 等资源的统一分配与管理，降低手动维护资源生命周期的复杂度。

### 5.6 host侧代码

接下来需要补充Host 侧核函数调用代码：`KernelAdd` ，负责在 Host 侧组织完整的核函数调用流程。其实现遵循异构计算的标准范式，主要包括以下七个步骤：

- 初始化 AscendCL 运行环境；
- 申请与管理运行时资源，包括 Device、Context 和 Stream；
- 在 Host 侧准备输入数据，并申请 Host 侧与 Device 侧内存，将数据从 Host 侧拷贝至 Device 侧；
- 通过核函数调用符 `<<<...>>>` 在指定 AI Core 上执行核函数；
- 将计算结果从 Device 侧内存拷贝回 Host 侧；
- 释放已申请的运行时资源；
- 去初始化 AscendCL 环境。

补充 `VerifyResult` 函数用于将 Ascend C 算子计算得到的输出结果 `output` 与 Host 侧按相同逻辑计算得到的期望结果 `golden` 进行比对，并输出验证结果。

补充`main` 函数负责定义输入数据长度、准备测试数据、调用算子并完成结果验证。


In [ ]:
%%writefile -a TPipeSources/add_tpipe_tque.asc

std::vector<float> KernelAdd(std::vector<float>& x, std::vector<float>& y)
{
    constexpr uint32_t numBlocks = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    uint8_t* xHost = reinterpret_cast<uint8_t*>(x.data());
    uint8_t* yHost = reinterpret_cast<uint8_t*>(y.data());
    uint8_t* zHost = nullptr;
    uint8_t* xDevice = nullptr;
    uint8_t* yDevice = nullptr;
    uint8_t* zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    aclrtMallocHost((void**)(&zHost), totalByteSize);
    aclrtMalloc((void**)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    add_custom<<<numBlocks, nullptr, stream>>>(xDevice, yDevice, zDevice, totalLength);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z(reinterpret_cast<float*>(zHost), reinterpret_cast<float*>(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);

    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return z;
}

uint32_t VerifyResult(std::vector<float>& output, std::vector<float>& golden)
{
    auto printTensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;

    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);
    std::vector<float> golden(totalLength, valueX + valueY);
    std::vector<float> output = KernelAdd(x, y);

    return VerifyResult(output, golden);
}

### 5.7 CMake 编译配置

一个完整的算子应用程序需要通过编译生成可执行文件。对于 Ascend C 项目，可以使用 CMake 作为构建系统，并通过特定配置调用 Ascend C 编译工具链。完整的编译配置与说明如下：

In [ ]:
%%writefile TPipeSources/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")
# find_package(ASC) 是 CMake 中用于查找和配置 Ascend C 编译工具链的命令
find_package(ASC REQUIRED)
# 指定项目支持的语言包括 ASC 和 CXX，ASC 表示支持使用毕昇编译器对 Ascend C 编程语言进行编译
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_tpipe_tque.asc
)
# 通过编译选项设置 NPU 架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

配置要点说明：

- 工具链依赖：核心是 `find_package(ASC)`，它会自动配置将 `.asc` 文件编译为 NPU 可执行代码所需的规则。
- 双语言支持：通过 `project(... LANGUAGES ASC CXX)` 声明工程同时包含 Ascend C 和 C++ 代码，CMake 会分别用正确的编译器处理 Host 侧 C++ 代码和 Device 侧 Ascend C 核函数代码。
- 架构指定：`--npu-arch` 选项必须与运行环境的 NPU 型号匹配，以确保生成的二进制指令能够正确执行。昇腾 AI 处理器型号和 `__NPU_ARCH__` 的对应关系如下表：

    <table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:300px;">昇腾 AI 处理器型号</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:100px;">__NPU_ARCH__</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas A2 训练系列产品/Atlas A2 推理系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2201</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Atlas A3 训练系列产品/Atlas A3 推理系列产品</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2201</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">Ascend 950PR/Ascend 950DT</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3510</td>
    </tr> 
    </table>

此配置提供了一个最小且完整的编译框架。在实际项目中，可能还需要添加头文件路径、链接库，以及针对不同构建类型（Debug/Release）的优化选项。


### 5.8 编译和运行

执行以下命令编译可执行文件：

In [ ]:
!cmake -S TPipeSources -B TPipeSources/build -DCMAKE_ASC_ARCHITECTURES=dav-2201  # 配置构建目录并指定 NPU 架构
!cmake --build TPipeSources/build -j  # 编译生成 demo 可执行文件

再执行以下代码，进行算子的实际运行：

In [ ]:
!cd TPipeSources/build && ./demo  

---

## 6. Q&A

1. 为什么 `DataCopy` 之后需要入队再马上出队？为什么不能直接使用 `xLocal`？`EnQue` 和 `DeQue` 不是多此一举吗？

    ```cpp
    AscendC::DataCopy(xLocal, xGm, blockLength);
    inQueueX.EnQue(xLocal);
    xLocal = inQueueX.DeQue<float>();
    ```

    在本节的 Add 样例里，`EnQue` 后面马上 `DeQue`，看起来像多此一举，是因为样例把 CopyIn、Compute、CopyOut 写在相邻代码中，逻辑上距离很近。但从框架语义看，这一步仍然是在表达阶段交接：CopyIn 阶段交付输入 tensor，Compute 阶段再从输入队列中领取它。这组接口表达的是 **CopyIn 到 Compute 的数据依赖**。

    - `DataCopy(xLocal, xGm, ...)` 将搬入任务发给数据搬运流水；
    - `EnQue(xLocal)` 标记 CopyIn 阶段产物，并建立同步信号，通知下游 `DeQue` 该 tensor 已经可以被消费；
    - `DeQue<float>()` 在 Compute 阶段等待该同步信号，确保 Vector 读取 `xLocal` 前，GM 到 UB 的搬入已经完成。

    所以，`EnQue` 和 `DeQue` 不是为了移动数据，而是为了建立阶段边界和同步关系。

2. `EnQue/DeQue` 会把搬入单元的数据传输到计算单元吗？

    不会。真正的数据传输由 `DataCopy` 完成。`DataCopy` 把数据写入 `LocalTensor` 对应的片上内存，例如 UB。Vector 计算单元后续从这块 UB 内存读取数据。

    `EnQue/DeQue` 的主要作用是：

    - 管理队列中 `LocalTensor` 的句柄或地址顺序；
    - 在底层插入同步事件，保证什么时候可以安全读取或写入。

    因此可以这样理解：`DataCopy` 负责搬数据，`Add` 等计算 API 负责算数据，`EnQue/DeQue` 负责通知和同步。



---

## 7. 本章小结

本章围绕 `TPipe/TQue` 讲解了 Ascend C SIMD 编程中基础流水的组织方式。对于一个矢量算子来说，单个数据分片通常需要经历 CopyIn、Compute、CopyOut 三个阶段；在 AI Core 上，这些阶段更接近于向不同硬件执行单元发射任务，因此必须正确处理阶段间同步和片上 buffer 的生命周期。

`TPipe` 负责统一管理队列 buffer 等 Device 端资源，`TQue` 负责在不同流水阶段之间传递 `LocalTensor` 并建立同步关系。实际编程时，可以按照 `InitBuffer -> AllocTensor -> EnQue -> DeQue -> DataCopy/Compute -> EnQue -> DeQue -> FreeTensor` 的逻辑使用 `TPipe/TQue` 框架编写算子。


---

## 8. 课后习题

下面几道题用于检查你对 `TPipe/TQue` 编程框架的理解。第 1 题是简答题，第 2、3 题是概念单选题，第 4、5 题用于判断常见错误代码。

### 1. 简答题：`TQue` 常用的几个核心 API 分别是什么？它们在 CopyIn、Compute、CopyOut 流水中的作用是什么？

请至少说明以下 4 个接口：`AllocTensor`、`EnQue`、`DeQue`、`FreeTensor`。

### 2. 关于 `TPipe` 的职责，下列说法正确的是？

A. `TPipe` 主要负责执行 `Add`、`Mul` 等矢量计算。  
B. `TPipe` 主要负责统一管理 Device 端内存等资源，例如为 `TQue` 分配 buffer，并管理部分同步事件资源。  
C. `TPipe` 主要负责把 Global Memory 中的数据搬运到 Local Memory。  
D. 一个 Kernel 中可以随意创建多个 `TPipe` 对象，它们之间互不影响。

### 3. 关于 `TQue` 的作用，下列说法正确的是？

A. `TQue` 是普通 C++ 队列，只保存 C++ 对象，不参与硬件同步。  
B. `TQue` 用于流水任务之间的通信和同步，例如 CopyIn 将数据入队，Compute 再从队列取出使用。  
C. `TQue` 负责真正执行 GM 与 UB 之间的数据搬运。  
D. `TQue` 只能用于输出数据，不能用于输入数据。

### 4. 下面代码的主要问题是什么？

```cpp
__aicore__ inline void pipe_init()
{
    AscendC::TPipe pipe;
}

__global__ __vector__ void test_kernel(__gm__ uint8_t* x, __gm__ uint8_t* y, __gm__ uint8_t* z,
                                       uint32_t totalLength)
{
    pipe_init();
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueY;
    uint32_t blockLength = totalLength / AscendC::GetBlockNum();
    pipe.InitBuffer(inQueueX, 1, blockLength * sizeof(float));
    pipe.InitBuffer(inQueueY, 1, blockLength * sizeof(float));
}
```

A. `pipe` 是 `pipe_init()` 内部局部变量，生命周期结束后在 `test_kernel` 中不可用，`TPipe` 应在 Kernel 入口创建或通过引用传入。  
B. `TQue<AscendC::TPosition::VECIN, 1>` 的队列深度必须设置为 0。  
C. `blockLength` 不能通过 `GetBlockNum()` 计算。  
D. 一个 `TPipe` 只能给一个 `TQue` 调用 `InitBuffer`。

### 5. 下面代码的主要问题是什么？

```cpp
AscendC::TPipe pipe;
AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
uint32_t blockLength = totalLength / AscendC::GetBlockNum();
pipe.InitBuffer(inQueueX, 1, blockLength * sizeof(float));
AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
AscendC::DataCopy(xLocal, xGm, blockLength);
xLocal = inQueueX.DeQue<float>();
```

A. `DeQue` 前没有 `EnQue`，队列中没有上游提交的 tensor。  
B. `DataCopy` 必须放在 `DeQue` 之后。  
C. `InitBuffer` 的第二个参数不能为 1。  
D. `xLocal` 不能重复赋值。

**执行以下代码获取参考答案。**


In [ ]:
!cat ./answer/03.03.06_answer.txt

完成这些问题后，你应当能独立阅读大多数基于 `TPipe` / `TQue` 的基础 SIMD 样例，并能判断一段代码中的资源管理和队列同步关系是否合理。


---

## 附录：参考资料

更多关于 `TPipe/TQue` 的原理、编程范式和 API 使用说明，请参考以下文档：

- [TPipe-TQue 框架编程原理](/data/hxj/code/2/asc-devkit_0616/docs/guide/编程指南/编程模型/AI-Core-SIMD编程/基于TPipe-TQue框架编程/TPipe-TQue框架编程原理.md)
- [TPipe-TQue 框架编程范式](/data/hxj/code/2/asc-devkit_0616/docs/guide/编程指南/编程模型/AI-Core-SIMD编程/基于TPipe-TQue框架编程/TPipe-TQue框架编程范式.md)
- [Pipe 和 Que 框架 API 说明](/data/hxj/code/2/asc-devkit_0616/docs/api/SIMD-API/基础API/资源管理/Pipe和Que框架/Pipe和Que框架.md)
